### **Technical Report**

1. **Problem description**
- The primary objective of the program was to optimize the processing of a dataset (Bureau of Customs 2019 lite data) using distributed computing and measure whether a parallelized approach using multiple processors could significantly reduce execution time compared to a standard sequential Python script. The activity focused on three specific data operations:
 - Filtering: Extracting records where the country of origin is "CHN".
 - Aggregation: Calculating the mean dutiable value grouped by country.
 - Sorting: Organizing the dataset by dutiable value in descending order.
 
2. **Parallelization approach**
- The parallelization approach of the program utilizes the Master-Worker architecture using mpi4py with 5 processes. The master loads the primary DataFrame, partitions the data into chunks, distributes tasks to workers, and gathers the results. The ffour other workers receive data chunks from the master, perform local filtering, aggregation, and sorting, then send the processed results back to the master.

3. **Performance analysis**

| # | Task | Sequential (seconds) | Parallel (seconds) |
| --- | ---- | -------------------- | ------------------ |
| 0	| Filter Data | 1.4150 | 0.0939 |
| 1	| Aggregate Data	| 0.0275	| 0.0030 |
| 2	| Sort Data	| 0.0588	| 0.3165 |
| 3 	| TOTAL	| 1.5013	| 0.4134 |

- Based on the summary of the execution logs, the parallel implementation showed a substantial overall performance gain with Parallel being faster than Sequential on the Filter and Aggregate functions, while showing that Parallel is slower than Sequential in the sorting function. This is likely due to communication overhead, where the master has to send chunks to 4 workers, then receive 4 sorted chunks back via MPI. The serialization and deserialization of DataFrames over MPI takes more time than just doing it sequentially. There is also a factor where when the master is receiving all sorted chunks, is still has to pd.concat + sort_values again on the combined result to get a globally sorted dataset.


4. **Challenges encountered**
- There are a few challenges that I encountered. First, running mpi4py requires a specific terminal environment and the mpiexec command, which cannot be executed directly within standard Jupyter cells without using subprocess or external scripts. Second, it was a bit clunky making the transition between the sequential and parallel function, so the comm.Barrier() was necessary to make sure that the sequential baseline was finished before the parallel processes began.